# 🌍 Earth AI Vision: SegFormer-B0 & YOLOv8n-seg Satellite Pipeline
### High-Precision Optical Rooftop Detection, Road Network & Vehicle Inference for Google Earth

> ⚡ **Hardware Acceleration Required:** Go to **Runtime** ➔ **Change runtime type** ➔ Select **T4 GPU**!

Click the **Play / Run button ▶️** on the cell below.
It will automatically:
1. Install all dependencies (`fastapi`, `mercantile`, `opencv`, `transformers`, `torch`, `ultralytics`).
2. Load SegFormer-B0 & YOLOv8n onto the Tesla T4 GPU.
3. Launch a free encrypted Cloudflare tunnel and print your live public URL:
   `🚀 LIVE PUBLIC COLAB API URL: https://xxxx.trycloudflare.com`

👉 Copy that URL and paste it into the **Colab Backend URL** in your Google Earth app!

In [ ]:
# 1. Install dependencies
!pip install -q fastapi uvicorn mercantile Pillow opencv-python transformers torch torchvision ultralytics

import os, time, subprocess, re, base64

# Clean up previous processes
!pkill -f "colab_server.py" || true
!pkill -f "cloudflared" || true
!fuser -k 8000/tcp || true

b64_code = """..."""

with open("colab_server.py", "wb") as f:
    f.write(base64.b64decode(b64_code))

print("✅ colab_server.py written successfully with Optical Rooftop & Multi-Tile Pipeline!")

# Download cloudflared if not present
if not os.path.exists("cloudflared"):
    subprocess.run(["wget", "-q", "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", "-O", "cloudflared"])
    subprocess.run(["chmod", "+x", "cloudflared"])

# Start Cloudflare tunnel in background
tunnel_proc = subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

print("[*] Establishing Cloudflare Tunnel...")
public_url = None
for _ in range(50):
    line = tunnel_proc.stdout.readline()
    if line:
        match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
        if match:
            public_url = match.group(0)
            break
    time.sleep(0.3)

print("=" * 65)
if public_url:
    print(f"🚀 LIVE PUBLIC COLAB API URL:  {public_url}")
    print("=" * 65)
    print("👉 Copy this URL and paste it into Earth AI config in your Google Earth app!")
else:
    print("⚠️ Tunnel starting. Check logs or rerun cell.")
print("=" * 65)

# Run server directly in foreground to display model initialization & request logs
!python3 colab_server.py
